# TNMRControl Clean Sequence Tables From One Messy TNT

This notebook starts from one messy source TNT only. It makes a new TNT copy first, then lets you clear table contents that do not appear to be used by the active sequence.

Important limitation: this does not delete table names. The documented TNMR OLE commands exposed in `TNMRControl` are `GetTableList`, `GetTable`, and `SetTable`; we do not have a verified delete-table command. If TNMR keeps the table name after `SetTable(name, "0\r\n")`, that is expected.

## 1. Setup

In [ ]:
from datetime import datetime
from pathlib import Path
import importlib
import sys

redstone_dir = next(p / "hardware_capture" / "rf" / "redstone" for p in (Path.cwd(), *Path.cwd().parents) if (p / "hardware_capture" / "rf" / "redstone").is_dir())
if str(redstone_dir) not in sys.path:
    sys.path.insert(0, str(redstone_dir))

import tnmrcontrol
importlib.reload(tnmrcontrol)

from tnmrcontrol import TNMRControl

## 2. Select The Messy Source TNT

Edit only `SOURCE_TNT`. The notebook derives output paths automatically and refuses to overwrite the source.

In [ ]:
SOURCE_TNT = Path(r"C:\TNMR\data\common\source_with_many_tables.tnt")

OUT_DIR = SOURCE_TNT.with_name("python_clean_tables")
OUT_DIR.mkdir(parents=True, exist_ok=True)

stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
EXPORTED_SEQUENCE = OUT_DIR / f"{SOURCE_TNT.stem}_sequence_{stamp}.tps"
OUTPUT_TNT = OUT_DIR / f"{SOURCE_TNT.stem}_cleared_unused_tables_{stamp}.tnt"

assert SOURCE_TNT.exists(), SOURCE_TNT
assert OUTPUT_TNT.resolve() != SOURCE_TNT.resolve()
assert not OUTPUT_TNT.exists(), OUTPUT_TNT

SOURCE_TNT, EXPORTED_SEQUENCE, OUTPUT_TNT

## 3. Open Source And Read Tables

This is read-only except for exporting the active sequence to a TPS file.

In [ ]:
source = TNMRControl()
source.connect(launch=False)
source.open(SOURCE_TNT)

snapshot = source.snapshot()
print("Source:", source.path)
print("Sequence:", snapshot.get("sequence_name"))
print("Warnings:", snapshot.get("warnings", []))

source_table_names = source.get_table_list()
source_tables = {}
table_errors = {}

for name in source_table_names:
    try:
        source_tables[name] = source.get_table(name)
    except Exception as exc:
        table_errors[name] = str(exc)

print("Tables listed:", len(source_table_names))
print("Tables read:", len(source_tables))
if table_errors:
    print("Unreadable tables:")
    for name, error in table_errors.items():
        print(" -", name, error)

source_table_names[:30]

## 4. Export Active Sequence

In [ ]:
source.save_sequence_as(EXPORTED_SEQUENCE)
assert EXPORTED_SEQUENCE.exists(), EXPORTED_SEQUENCE
print("Exported sequence:", EXPORTED_SEQUENCE)

## 5. Find Candidate Used Tables

TPS files are binary, but table names appear as strings. This helper searches for current table names and suppresses obvious table-definition occurrences. Treat the output as a candidate list; review before clearing anything.

In [ ]:
def find_all_offsets(data, needle):
    offsets = []
    start = 0
    while True:
        offset = data.find(needle, start)
        if offset < 0:
            return offsets
        offsets.append(offset)
        start = offset + max(1, len(needle))


def printable_window(data, offset, radius=90):
    chunk = data[max(0, offset - radius): offset + radius]
    chars = []
    for byte in chunk:
        if 32 <= byte < 127:
            chars.append(chr(byte))
        elif byte in (9, 10, 13):
            chars.append(" ")
        else:
            chars.append(".")
    return "".join(chars)


def looks_like_table_definition(data, offset):
    nearby = data[offset: offset + 512]
    return b"+ Add" in nearby or b"Every pass" in nearby


def candidate_table_references(tps_path, table_names):
    data = Path(tps_path).read_bytes()
    candidates = {}
    all_occurrences = {}
    for name in table_names:
        needle = str(name).encode("latin-1", errors="ignore")
        if not needle:
            continue
        offsets = find_all_offsets(data, needle)
        if not offsets:
            continue
        all_occurrences[name] = offsets
        reference_offsets = [
            offset for offset in offsets if not looks_like_table_definition(data, offset)
        ]
        if reference_offsets:
            candidates[name] = reference_offsets
    return data, candidates, all_occurrences

In [ ]:
tps_data, candidate_offsets, all_offsets = candidate_table_references(
    EXPORTED_SEQUENCE,
    source_table_names,
)

candidate_used = sorted(candidate_offsets)
candidate_unused = sorted(set(source_table_names) - set(candidate_used))

print("All source tables:", len(source_table_names))
print("Candidate used tables:", len(candidate_used))
print("Candidate unused tables:", len(candidate_unused))
print("\nCandidate used:")
for name in candidate_used:
    print(" -", name)

print("\nCandidate unused preview:")
for name in candidate_unused[:30]:
    print(" -", name)

## 6. Review Candidate Context

In [ ]:
for name in candidate_used:
    print("\n===", name, "===")
    for offset in candidate_offsets[name][:3]:
        print("offset", offset)
        print(printable_window(tps_data, offset))

## 7. Choose Tables To Keep

Edit `USED_TABLE_NAMES` after reviewing the candidate list. Anything not listed here is considered unused for the clearing step.

In [ ]:
USED_TABLE_NAMES = candidate_used.copy()

# Example manual override:
# USED_TABLE_NAMES = ["phi", "phacq"]

missing = sorted(set(USED_TABLE_NAMES) - set(source_tables))
assert not missing, missing

TABLES_TO_CLEAR = sorted(set(source_table_names) - set(USED_TABLE_NAMES))

print("Will keep", len(USED_TABLE_NAMES), "tables:")
for name in USED_TABLE_NAMES:
    print(" -", name)

print("\nWill clear", len(TABLES_TO_CLEAR), "tables:")
for name in TABLES_TO_CLEAR[:50]:
    print(" -", name)

## 8. Save A Copy First

This writes a new TNT copy and changes the open document path to that copy. The source file path is protected by assertions.

In [ ]:
CREATE_COPY = False

assert CREATE_COPY, "Review paths and set CREATE_COPY = True."
assert OUTPUT_TNT.resolve() != SOURCE_TNT.resolve()
assert not OUTPUT_TNT.exists(), OUTPUT_TNT

source.save_as(OUTPUT_TNT)
assert OUTPUT_TNT.exists(), OUTPUT_TNT
print("Created working copy:", OUTPUT_TNT)

## 9. Clear Unused Table Contents In The Copy

This uses `SetTable(name, "0\r\n")` on tables you chose not to keep. TNMR may still list those table names afterwards. This is a content cleanup, not a guaranteed table-name deletion.

In [ ]:
CLEAR_UNUSED_TABLES = False
CLEAR_VALUE = "0\r\n"

assert CLEAR_UNUSED_TABLES, "Review TABLES_TO_CLEAR and set CLEAR_UNUSED_TABLES = True."
assert source.path == str(OUTPUT_TNT.resolve()), "Run the Save A Copy cell first."

for name in TABLES_TO_CLEAR:
    source.set_table(name, CLEAR_VALUE)

source.save()
print("Cleared", len(TABLES_TO_CLEAR), "unused table contents in:", OUTPUT_TNT)

## 10. Verify The Copy

In [ ]:
verify = TNMRControl()
verify.connect(launch=False)
verify.open(OUTPUT_TNT)

verify_tables = verify.get_table_list()
print("Verified TNT:", verify.path)
print("Sequence:", verify.sequence_name())
print("Tables still listed by TNMR:", len(verify_tables))

print("\nKept table previews:")
for name in USED_TABLE_NAMES:
    print("\n---", name, "---")
    print(str(verify.get_table(name))[:500])

print("\nCleared table previews:")
for name in TABLES_TO_CLEAR[:20]:
    print("\n---", name, "---")
    print(str(verify.get_table(name))[:120])